# 이커머스 고객 이탈 예측 — 전처리 & Train/Val/Test 분리

`make_snapshot()`으로 생성한 고객 스냅샷을 기반으로 Train/Val/Test를 분리하고,
전처리 Pipeline(변환기)을 구성한다.

**역할 범위**: 본 노트북은 전처리 완료 데이터와 전처리 Pipeline을 준비하는 것까지이며,
모델 학습·평가는 팀원이 이어받아 진행한다. 따라서 여기서는 모델 학습 없이
Train/Val/Test 분리 → 전처리 규칙 정의 → 변환기 저장까지만 수행한다.

- Test는 최종 평가 1회에만 사용 (가이드 요구) — 분리 이후 팀원도 재분리하지 않고 그대로 사용
- 평가는 이탈(1) 클래스의 Recall 우선 (전처리 자체와는 무관하나 팀 공유용으로 명시)

In [2]:
import sys
sys.path.append("..")
import pandas as pd
from src.features import make_snapshot

snapshot = make_snapshot(pd.Timestamp("2011-09-10"), window=90)
print(snapshot.shape)

(4320, 14)


In [3]:
from sklearn.model_selection import train_test_split

feature_cols = ["recency_days", "frequency", "distinct_products",
                 "net_revenue", "tenure_days", "is_low_value", "is_uk"]
target_col = "churn"

X = snapshot[feature_cols]
y = snapshot[target_col]

RANDOM_STATE = 42  # 팀 전체 공통 시드 — 절대 변경 금지

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=RANDOM_STATE
)

for name, arr in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"{name}: {len(arr)}행, 이탈률 {arr.mean():.3f}")

Train: 2592행, 이탈률 0.494
Val: 864행, 이탈률 0.494
Test: 864행, 이탈률 0.494


In [7]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline

log_cols = ["net_revenue"]
scale_cols = ["recency_days", "frequency", "distinct_products", "tenure_days"]
passthrough_cols = ["is_low_value", "is_uk"]

log_scale_pipe = Pipeline([
    ("clip", FunctionTransformer(lambda x: np.clip(x, 0, None), validate=True)),
    ("log", FunctionTransformer(np.log1p, validate=True)),
    ("scale", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("log_scale", log_scale_pipe, log_cols),
    ("scale", StandardScaler(), scale_cols),
    ("passthrough", "passthrough", passthrough_cols),
])

# Train으로만 fit (Val/Test는 transform만)
preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape, X_val_processed.shape, X_test_processed.shape)

(2592, 7) (864, 7) (864, 7)


### 예외 처리: net_revenue 음수값

7명의 고객(Train 5, Val 1, Test 1)에서 순매출이 음수로 나타났다(최소 -1192.20).
관찰구간 내 취소가 원 구매보다 많거나 취소만 단독으로 발생한 경우로 추정된다
(1부 EDA에서 확인한 취소-구매 쌍 문제의 연장선).

로그 변환 전 0으로 클리핑(clip)하여 처리한다 — "실질적으로 판매가 없었다"는
의미로 해석하며, 정보 손실 없이 로그 계산 오류(NaN)를 방지한다.

## 전처리 완료 데이터 저장

Train/Val/Test와 학습된 전처리 Pipeline을 파일로 저장한다.
팀원들은 이 파일들을 그대로 불러와 모델 학습을 시작하면 되며,
전처리 과정을 재실행할 필요가 없다.

In [ ]:
import pickle
from pathlib import Path

OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 원본 스케일 X (참고용, 컬럼명 유지)
X_train.to_csv(OUT_DIR / "X_train.csv", index=False)
X_val.to_csv(OUT_DIR / "X_val.csv", index=False)
#X_test.to_csv(OUT_DIR / "X_test.csv", index=False)

y_train.to_csv(OUT_DIR / "y_train.csv", index=False)
y_val.to_csv(OUT_DIR / "y_val.csv", index=False)
#y_test.to_csv(OUT_DIR / "y_test.csv", index=False)

# 전처리된(변환 완료) 버전도 저장 — 바로 모델에 넣을 수 있게
pd.DataFrame(X_train_processed, columns=feature_cols).to_csv(OUT_DIR / "X_train_processed.csv", index=False)
pd.DataFrame(X_val_processed, columns=feature_cols).to_csv(OUT_DIR / "X_val_processed.csv", index=False)
pd.DataFrame(X_test_processed, columns=feature_cols).to_csv(OUT_DIR / "X_test_processed.csv", index=False)

# 전처리 Pipeline 저장 (팀원이 새 데이터에 동일 변환 적용할 때 사용)
with open(OUT_DIR / "preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)

print("저장 완료:", list(OUT_DIR.glob("*")))